In [19]:
import torch
import torch.nn as nn
import torch.multiprocessing as mp
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import warnings

warnings.filterwarnings('ignore')
np.random.seed(42)

print('Imports successful.')

Imports successful.


In [20]:
# Load features
X = np.load('../data/X_features.npy')
y = np.load('../data/y_labels.npy')

# Encode labels to numbers
le = LabelEncoder()
y_enc = le.fit_transform(y)

print(f'Classes: {le.classes_}')
print(f'Encoded: {np.unique(y_enc)}')

Classes: ['angry' 'calm' 'disgust' 'fearful' 'happy' 'neutral' 'sad' 'surprised']
Encoded: [0 1 2 3 4 5 6 7]


In [21]:
# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)

print(f'Train size: {X_train.shape[0]}')
print(f'Test size:  {X_test.shape[0]}')
print(f'Feature size: {X_train.shape[1]}')

Train size: 2304
Test size:  576
Feature size: 112


In [22]:
from concurrent.futures import ProcessPoolExecutor, as_completed
from parallel_workers import run_model

jobs = ['svm', 'rf']

if __name__ == '__main__':
    # Use ProcessPoolExecutor to run models in parallel.
    with ProcessPoolExecutor(max_workers=8) as executor:
        futures = [
            executor.submit(run_model, job, X_train, y_train, X_test, y_test)
            for job in jobs
        ]

        results = [future.result() for future in as_completed(futures)]

    print('\nSummary:')
    for index, result in enumerate(results, start=1):
        print(
            f"{index}. {result['model'].upper()} -> "
            f"accuracy={result['accuracy']:.4f}, "
            f"precision={result['precision']:.4f}, "
            f"recall={result['recall']:.4f}, "
            f"f1={result['f1']:.4f}"
        )


Summary:
1. SVM -> accuracy=0.9583, precision=0.9589, recall=0.9583, f1=0.9585
2. RF -> accuracy=0.9167, precision=0.9183, recall=0.9167, f1=0.9163
